In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that performs sparse matrix-vector multiplication.
  Given a sparse matrix $A$ of dimensions $M \times N$ and a dense vector $x$ of length $N$,
  compute the product vector $y = A \times x$, which will have length $M$. <code>A</code> is stored in row-major order.
  <code>nnz</code> is the number of non-zero elements in <code>A</code>.
</p>

<p>
  Mathematically, the operation is defined as:
  $$
  y_i = \sum_{j=0}^{N-1} A_{ij} \cdot x_j \quad \text{for} \quad i = 0, 1, \ldots, M-1
  $$
</p>

<p>
  The matrix $A$ is approximately 60 - 70% sparse.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only GPU native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in vector <code>y</code></li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
Matrix $A$ ($3 \times 4$):
$$
\begin{bmatrix}
5.0 & 0.0 & 0.0 & 1.0 \\
0.0 & 2.0 & 3.0 & 0.0 \\
0.0 & 0.0 & 0.0 & 4.0
\end{bmatrix}
$$
Vector $x$:
$$
\begin{bmatrix}
1.0 \\
2.0 \\
3.0 \\
4.0
\end{bmatrix}
$$
Output:<br>
Vector $y$:
$$
\begin{bmatrix}
9.0 \\
13.0 \\
16.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 10,000</li>
  <li>The matrix $A$ is approximately 60-70% sparse (i.e., 60-70% of elements are zero)</li>

  <li>Performance is measured with <code>M</code> = 1,000, <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, x, y are device pointers
extern "C" void solve(const float* A, const float* x, float* y, int M, int N, int nnz) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, x, y are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor, x: cute.Tensor, y: cute.Tensor, M: cute.Int32, N: cute.Int32, nnz: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A, x are tensors on the GPU
@jax.jit
def solve(A: jax.Array, x: jax.Array, M: int, N: int, nnz: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float32, MutExternalOrigin],
    x: UnsafePointer[Float32, MutExternalOrigin],
    y: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
    nnz: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, x, y are tensors on the GPU
def solve(A: torch.Tensor, x: torch.Tensor, y: torch.Tensor, M: int, N: int, nnz: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# A, x, y are tensors on the GPU
def solve(A: torch.Tensor, x: torch.Tensor, y: torch.Tensor, M: int, N: int, nnz: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/18_sparse_matrix_vector_multiplication/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
